# Pip installs

In [1]:
#pip install pandas
#pip install scikit-learn
#pip install tqdm
#pip install nltk
#pip install PyPDF2
#pip install spacy
#pip install torch
#pip install transformers
#pip install nbformat 

SyntaxError: invalid syntax (308230526.py, line 1)

# Imports

In [5]:
import os
import pandas as pd
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
import PyPDF2
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import unicodedata
import re
from sklearn.feature_extraction.text import CountVectorizer
import spacy
from collections import Counter
from transformers import BertForSequenceClassification, BertTokenizer
import torch

ModuleNotFoundError: No module named 'torch'

### One-time downloads

In [3]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
spacy.cli.download('nl_core_news_lg')
spacy.cli.download('en_core_web_lg')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\daanv\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\daanv\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\daanv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\daanv\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\daanv\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\daanv\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunke

✔ Download and installation successful
You can now load the package via spacy.load('nl_core_news_lg')
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


### Dutch or English

In [4]:
# Load Spacy and stop words Dutch
nlp = spacy.load('nl_core_news_lg')
stop_words = set(stopwords.words('dutch'))

# Load Spacy and stop words English
# nlp = spacy.load('en_core_web_lg')
# english_stop_words = set(stopwords.words('english'))

# Read PDF

In [5]:
def read_pdf(pdf_file, priority, exclude_pages=[1,2,3,4,5,22,23,24,25,26,27,87], filter_amount=20): # Adjust if needed
    if exclude_pages is None:
        exclude_pages = []

    # Adjust for 0-based indexing
    exclude_pages = [page - 1 for page in exclude_pages]

    all_text = ""
    page_numbers = []

    try:
        with open(pdf_file, "rb") as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = len(pdf_reader.pages)

            # Iterate through pages except the last 'filter_amount' pages
            for page_num in range(num_pages - filter_amount):
                # Skip pages in the exclude_pages list
                if page_num in exclude_pages:
                    print(f"Skipping excluded page {page_num + 1} of {pdf_file}.")
                    continue

                page = pdf_reader.pages[page_num]
                page_text = page.extract_text()

                # Character check
                if len(page_text) < 60: # Adjust if needed
                    print(f"Skipping page {page_num + 1} of {pdf_file} with {len(page_text)} characters.")
                    continue

                all_text += page_text

                # Extract page number for each sentence on this page
                sentences = sent_tokenize(page_text)
                num_sentences = len(sentences)

                for _ in range(num_sentences):
                    page_numbers.append(page_num + 1)

            # Split text into sentences
            sentences = sent_tokenize(all_text)
            # Add sentences and their corresponding page numbers to DataFrame
            df_data = {'Phrase': sentences, 'Page': page_numbers[:len(sentences)], 'Priority': [priority]*len(sentences)}
            df = pd.DataFrame(df_data)
            return df

    except Exception as e:
        print(f"Error reading {pdf_file}: {e}")
        return pd.DataFrame()

# Text Preprocessing

In [6]:
def restore_split_words(sentence, nlp, stopwords):
    doc = nlp(sentence)
    new_tokens = []
    skip_next = False
    restored_words = []  # List to keep track of restored words for debugging

    for i, token in enumerate(doc[:-1]):  # Loop through tokens except for the last one
        if skip_next:
            skip_next = False
            continue
        next_token = doc[i + 1]
        # Check if either token is a stopword or a single character
        if (token.text.lower() in stopwords or next_token.text.lower() in stopwords or
            len(token.text) == 1 or len(next_token.text) == 1):
            new_tokens.append(token.text)  # Append first token only, skip combining
        else:
            combined_text = token.text + next_token.text
            if nlp.vocab[combined_text].is_oov:  # Check if the combined text is out-of-vocabulary
                new_tokens.append(token.text)
            else:
                new_tokens.append(combined_text)
                restored_words.append((token.text, next_token.text, combined_text))  # Add the original and restored word to the list
                skip_next = True  # Skip the next token since it has been combined with the current one
    if not skip_next and doc:  # Append the last token if it wasn't combined
        new_tokens.append(doc[-1].text)

    sentence_after = ' '.join(new_tokens)  # Reconstruct sentence from new tokens

    if restored_words:  # Print restoration changes if any
        print("Restoration changes:")
        for original1, original2, combined in restored_words:
            print(f"{original1} + {original2} -> {combined}")
            
    return sentence_after

def process_text(df):
    processed_data = {'Phrase': [], 'Page': [], 'Priority': []}
    for _, row in df.iterrows():
        sentence = row['Phrase']

        sentence = unicodedata.normalize('NFKD', sentence)
        sentence = ''.join([c for c in sentence if not unicodedata.combining(c)])
        sentence = sentence.replace('\n', '').lower()
        sentence = re.sub(r'[^a-zA-Z\s]', '', sentence)
        sentence = sentence.replace("rightarrow", "").replace("leftarrow", "")

        # Restore potentially split words before removing stopwords
        restored_sentence = restore_split_words(sentence, nlp, stop_words)

        # Named entity recognition (NER) to filter out personal names
        doc = nlp(restored_sentence)  # Rebuild sentence for NER
        filtered_names = [ent.text for ent in doc.ents if ent.label_ == 'PERSON']
        if filtered_names:
            print("Filtered out personal names:", filtered_names)
        for name in filtered_names:
            restored_sentence = restored_sentence.replace(name, '')

        words = word_tokenize(restored_sentence)
        processed_words = [w for w in words if w.lower() not in stop_words and len(w) < 30 and len(w) > 1]

        processed_data['Phrase'].append(processed_words)  # Store processed words as a list
        processed_data['Page'].append(row['Page'])
        processed_data['Priority'].append(row['Priority'])

    processed_df = pd.DataFrame(processed_data)
    return processed_df

# Word Pair Generating

In [7]:
def generate_word_pairs(df):
    # Define the inner function to generate word pairs for each phrase
    def generate_pairs(tokens):
        pairs = list(combinations(tokens, 2))  # Generate all possible combinations of word pairs
        return [(word1, word2) for word1, word2 in pairs if word1 != word2]

    # Apply the generate_pairs function to each phrase to generate word pairs
    df['word_pairs'] = df['Phrase'].apply(generate_pairs)

    # Flatten the list of word pairs
    flat_pairs = [pair for sublist in df['word_pairs'] for pair in sublist]

    # Count occurrences of each word pair
    word_pair_counts = Counter(flat_pairs)

    # Create a DataFrame to store word pairs and their counts
    word_pair_df = pd.DataFrame(word_pair_counts.items(), columns=['Word Pair', 'Count'])
    word_pair_df['Word1'] = word_pair_df['Word Pair'].apply(lambda x: x[0])
    word_pair_df['Word2'] = word_pair_df['Word Pair'].apply(lambda x: x[1])

    # Concatenate Word1 and Word2 columns to create a new column for sorting and eliminate duplicates
    word_pair_df['Sorted Pair'] = word_pair_df.apply(lambda x: tuple(sorted([x['Word1'], x['Word2']])), axis=1)
    word_pair_df = word_pair_df.drop_duplicates(subset='Sorted Pair').drop(columns='Sorted Pair')

    return word_pair_df[['Word Pair', 'Word1', 'Word2', 'Count']]


# Predict Category

In [8]:
def predict_category(word_df, Label, batch_size=256):
    label_mapping = {
        0: 'Water',
        1: 'Mobiliteit',
        2: 'Duurzaamheid',
        3: 'Overige'
    }

    # Setup GPU/CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Using device: {device}')

    # Load model and tokenizer
    tokenizer = BertTokenizer.from_pretrained("wietsedv/bert-base-dutch-cased")
    model = BertForSequenceClassification.from_pretrained('model_6_bertje_df_all')
    model.to(device)  # Move model to GPU

    word1_predictions = []
    word2_predictions = []

    num_batches = (len(word_df) + batch_size - 1) // batch_size

    for i in range(0, len(word_df), batch_size):
        batch_df = word_df.iloc[i:i+batch_size]

        tokens_word1 = tokenizer(batch_df['Word1'].tolist(), padding=True, truncation=True, return_tensors='pt').to(device)
        tokens_word2 = tokenizer(batch_df['Word2'].tolist(), padding=True, truncation=True, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs_word1 = model(**tokens_word1)
            outputs_word2 = model(**tokens_word2)

        probabilities_word1 = torch.softmax(outputs_word1.logits, dim=1)
        probabilities_word2 = torch.softmax(outputs_word2.logits, dim=1)

        for j in range(len(batch_df)):
            predicted_label_index_word1 = torch.argmax(probabilities_word1[j], dim=0).item()
            predicted_label_word1 = label_mapping[predicted_label_index_word1]
            word1_predictions.append(predicted_label_word1)

            predicted_label_index_word2 = torch.argmax(probabilities_word2[j], dim=0).item()
            predicted_label_word2 = label_mapping[predicted_label_index_word2]
            word2_predictions.append(predicted_label_word2)

        # Print progress
        progress = (i // batch_size) + 1
        print(f'Processed batch {progress}/{num_batches}')

    result_df = pd.DataFrame({
        'Word_Pair': word_df['Word Pair'],
        'Word1': word_df['Word1'],
        'Word1_Category': word1_predictions,
        'Word2': word_df['Word2'],
        'Word2_Category': word2_predictions,
        'Count': word_df['Count']
    })

    # Filter rows where either Word1 or Word2 matches the specified label
    result_df = result_df[(result_df['Word1_Category'] == Label) | (result_df['Word2_Category'] == Label)]

    return result_df


# Top Word Selection with TF-IDF

In [9]:
class PriorityTfidfVectorizer(TfidfVectorizer):
    def __init__(self, priorities=None, input='content', encoding='utf-8', decode_error='strict', strip_accents=None, lowercase=True, preprocessor=None, tokenizer=None, stop_words=None, token_pattern='(?u)\\b\\w\\w+\\b', ngram_range=(1, 1), analyzer='word', max_df=1.0, min_df=1, max_features=None, vocabulary=None, binary=False, dtype=float, norm='l2', use_idf=True, smooth_idf=True, sublinear_tf=False):
        super().__init__(input=input, encoding=encoding, decode_error=decode_error, strip_accents=strip_accents, lowercase=lowercase, preprocessor=preprocessor, tokenizer=tokenizer, stop_words=stop_words, token_pattern=token_pattern, ngram_range=ngram_range, analyzer=analyzer, max_df=max_df, min_df=min_df, max_features=max_features, vocabulary=vocabulary, binary=binary, dtype=dtype, norm=norm, use_idf=use_idf, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf)
        self.priorities = priorities

    def fit_transform(self, raw_documents, y=None):
        tfidf_matrix = super().fit_transform(raw_documents, y)
        # Multiply tf-idf scores by the priority of the document
        for i, priority in enumerate(self.priorities):
            tfidf_matrix[i, :] *= priority
        return tfidf_matrix

# Create TF-IDF vectors
def create_tfidf_vectors(documents, priorities):
    vectorizer = PriorityTfidfVectorizer(priorities=priorities)
    tfidf_matrix = vectorizer.fit_transform(documents)
    feature_names = vectorizer.get_feature_names_out()
    return tfidf_matrix, feature_names

def select_top_words_by_tfidf(processed_df, top_n=300):
    # Join the phrases by page number into single strings
    grouped_text = processed_df.groupby('Page')['Phrase'].apply(lambda x: ' '.join(map(str, x))).tolist()
    priorities = processed_df['Priority'].unique()
    
    # Calculate the TF-IDF matrix with priorities
    tfidf_matrix, feature_names = create_tfidf_vectors(grouped_text, priorities)
    
    # Create a DataFrame with the scores per page
    weightdf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
    weightdf = weightdf.T  # Transpose to have words as rows
    weightdf['TF-IDF_max'] = weightdf.max(axis=1)
    
    # Sort the DataFrame by TF-IDF_max in descending order
    sorted_weightdf = weightdf.sort_values(by='TF-IDF_max', ascending=False)
    
    # Select the top N words
    top_words = sorted_weightdf.head(top_n).index.tolist()
    return top_words


# Top Word Selection with TF-IDF with Filter for categorisation

In [10]:
def select_top_words_by_tfidf_cat(processed_df, word_df, top_n=300): # Adjust top_n if needed
    # Join the phrases by page number into single strings
    grouped_text = processed_df.groupby('Page')['Phrase'].apply(lambda x: ' '.join(map(str, x))).tolist()
    priorities = processed_df['Priority'].unique()
    
    # Calculate the TF-IDF matrix with priorities
    tfidf_matrix, feature_names = create_tfidf_vectors(grouped_text, priorities)
    
    # Create a DataFrame with the scores per page
    weightdf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
    weightdf = weightdf.T  # Transpose to have words as rows
    weightdf['TF-IDF_max'] = weightdf.max(axis=1)
    
    # Sort the DataFrame by TF-IDF_max in descending order
    sorted_weightdf = weightdf.sort_values(by='TF-IDF_max', ascending=False)
    
    top_words_filtered = sorted_weightdf[sorted_weightdf.index.isin(word_df['Word1']) | sorted_weightdf.index.isin(word_df['Word2'])]
    # Select the top N words
    top_words = top_words_filtered.head(top_n).index.tolist()
    return top_words


# Social Graph Constructing

In [11]:
def create_social_graph_from_top_words(word_df, top_words, processed_df):
    social_graph_df = word_df[(word_df['Word1'].isin(top_words)) & (word_df['Word2'].isin(top_words))].sort_values('Word1')

    # Calculate occurrence count for each word, considering priorities
    list_of_words = list(set(list(social_graph_df['Word1'].unique()) + list(social_graph_df['Word2'].unique())))
    occurrence_df = pd.DataFrame({'Word': list_of_words, 'Occurrence': [
        sum(word in tokens for tokens, priority in zip(processed_df['Phrase'], processed_df['Priority']) for _ in range(int(priority))) for word in list_of_words
    ]})
    # Merge occurrences back into the social graph DataFrame
    social_graph_df = pd.merge(social_graph_df, occurrence_df, left_on='Word1', right_on='Word', how='left').rename(columns={'Occurrence': 'Word1_occurrence'}).drop('Word', axis=1)
    social_graph_df = pd.merge(social_graph_df, occurrence_df, left_on='Word2', right_on='Word', how='left').rename(columns={'Occurrence': 'Word2_occurrence'}).drop('Word', axis=1)
    
    # Calculate weighted count using the PMI formula 
    social_graph_df['Count_weighted'] = social_graph_df['Count'] / (social_graph_df['Word1_occurrence'] * social_graph_df['Word2_occurrence'])
    
    return social_graph_df

# Main function

In [12]:
def main(pdf_files, doc_weights):
    df_data = {'Phrase': [], 'Page': [], 'Priority': []}  # Initialize df_data dictionary
    for pdf_file in pdf_files:
        if os.path.exists(pdf_file):
            text_df = read_pdf(pdf_file, doc_weights[pdf_file])
            if text_df is not None:  # Check if read_pdf returned a valid DataFrame
                df_data['Phrase'].extend(text_df['Phrase'])
                df_data['Page'].extend(text_df['Page'])
                df_data['Priority'].extend(text_df['Priority'])
            else:
                print(f"Error reading text from {pdf_file}")
        else:
            print(f"File '{pdf_file}' does not exist.")
    
    df = pd.DataFrame(df_data)  # Create DataFrame from df_data
    processed_df = process_text(df)  # Process the text DataFrame
    word_df = generate_word_pairs(processed_df)  # Generate word pairs df on processed data
    
    # Select the top 300 words based on TF-IDF scores
    top_n_words = select_top_words_by_tfidf(processed_df)
    
    # Create the social graph from the top 300 words
    social_graph_df = create_social_graph_from_top_words(word_df, top_n_words, processed_df)
    
    return social_graph_df, processed_df

# Main function with categorisation

In [13]:
def main_cat(pdf_files, doc_weights, Label):
    df_data = {'Phrase': [], 'Page': [], 'Priority': []}  # Initialize df_data dictionary
    for pdf_file in pdf_files:
        if os.path.exists(pdf_file):
            text_df = read_pdf(pdf_file, doc_weights[pdf_file])
            if text_df is not None:  # Check if read_pdf returned a valid DataFrame
                df_data['Phrase'].extend(text_df['Phrase'])
                df_data['Page'].extend(text_df['Page'])
                df_data['Priority'].extend(text_df['Priority'])
            else:
                print(f"Error reading text from {pdf_file}")
        else:
            print(f"File '{pdf_file}' does not exist.")
    

    df = pd.DataFrame(df_data)  # Create DataFrame from df_data
    processed_df = process_text(df)  # Process the text DataFrame
    word_df = generate_word_pairs(processed_df)  # Generate word pairs df on processed data
        

    # Choices are 'Water', 'Mobiliteit', 'Duurzaamheid', 'Armoede', 'Overige'
    word_df_cat = predict_category(word_df, Label)
    
    # Select the top 300 words based on TF-IDF scores
    top_n_words = select_top_words_by_tfidf_cat(processed_df, word_df_cat)

    
    social_graph = create_social_graph_from_top_words(word_df_cat, top_n_words, processed_df)

    return social_graph, processed_df

In [14]:
# List of PDF files to process
#pdf_files = ["presentatie_gebiedsvisie.pdf", "Ontwikkeldocument+NPLG_PDF.pdf"]
pdf_files = ["INEK.pdf"]
#pdf_files = [r"C:\Users\rosal\DataspellProjects\VanWonen\Notebooks\Kennisgrafiek\1.pdf"]



#rosalie
#pdf_files = [r"C:\Users\rosal\DataspellProjects\VanWonen\Notebooks\Kennisgrafiek\1.pdf",r"C:\Users\rosal\DataspellProjects\VanWonen\Notebooks\Werk\Final_models\Ontwikkeldocument+NPLG_PDF.pdf"]
# pdf_files = ["Armoede- en Schuldenmonitor_2023_DEF 27-9.pdf"]

In [15]:
# Define the document weights
doc_weights = {"INEK.pdf": 1.0}

#doc_weights = {r"C:\Users\rosal\DataspellProjects\VanWonen\Notebooks\Kennisgrafiek\1.pdf": 1.0}


#rosalie
#doc_weights = {r"C:\Users\rosal\DataspellProjects\VanWonen\Notebooks\Kennisgrafiek\1.pdf": 1.0, r"C:\Users\rosal\DataspellProjects\VanWonen\Notebooks\Werk\Final_models\Ontwikkeldocument+NPLG_PDF.pdf" : 1.0}

# doc_weights = {"Armoede- en Schuldenmonitor_2023_DEF 27-9.pdf": 1.0}

300 characters seems to be the 'golden number' in this case, this was determined through trial and error. When inspecting the document and checking the removed pages, setting the threshold at 300 removes every unwanted page while leaving every important page intact.

In [16]:
# Call the main function
# relations_df, df_data = main(pdf_files, doc_weights)

# Call the main function with category
relations_df, df_data = main_cat(pdf_files,doc_weights, 'Duurzaamheid')

Skipping excluded page 1 of INEK.pdf.
Skipping excluded page 2 of INEK.pdf.
Skipping excluded page 3 of INEK.pdf.
Skipping excluded page 4 of INEK.pdf.
Skipping excluded page 5 of INEK.pdf.
Skipping excluded page 22 of INEK.pdf.
Skipping excluded page 23 of INEK.pdf.
Skipping excluded page 24 of INEK.pdf.
Skipping excluded page 25 of INEK.pdf.
Skipping excluded page 26 of INEK.pdf.
Skipping excluded page 27 of INEK.pdf.
Skipping page 86 of INEK.pdf with 58 characters.
Skipping excluded page 87 of INEK.pdf.
Restoration changes:
ten + opzichte -> tenopzichte
Restoration changes:
vijf + jaar -> vijfjaar
Restoration changes:
korte + termijn -> kortetermijn
Restoration changes:
tussen + partijen -> tussenpartijen
Restoration changes:
ten + minste -> tenminste
ten + opzichte -> tenopzichte
Restoration changes:
ten + opzichte -> tenopzichte
Restoration changes:
be + eindigd -> beeindigd
Restoration changes:
ten + opzichte -> tenopzichte
Restoration changes:
ten + opzichte -> tenopzichte
Resto

C:\Users\naomi\anaconda3\Lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Processed batch 1/744
Processed batch 2/744
Processed batch 3/744
Processed batch 4/744
Processed batch 5/744
Processed batch 6/744
Processed batch 7/744
Processed batch 8/744
Processed batch 9/744
Processed batch 10/744
Processed batch 11/744
Processed batch 12/744
Processed batch 13/744
Processed batch 14/744
Processed batch 15/744
Processed batch 16/744
Processed batch 17/744
Processed batch 18/744
Processed batch 19/744
Processed batch 20/744
Processed batch 21/744
Processed batch 22/744
Processed batch 23/744
Processed batch 24/744
Processed batch 25/744
Processed batch 26/744
Processed batch 27/744
Processed batch 28/744
Processed batch 29/744
Processed batch 30/744
Processed batch 31/744
Processed batch 32/744
Processed batch 33/744
Processed batch 34/744
Processed batch 35/744
Processed batch 36/744
Processed batch 37/744
Processed batch 38/744
Processed batch 39/744
Processed batch 40/744
Processed batch 41/744
Processed batch 42/744
Processed batch 43/744
Processed batch 44/7

C:\Users\naomi\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:2072: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. <class 'float'> 'dtype' will be converted to np.float64.
  warnings.warn(


In [17]:
relations_df.head()

,Word_Pair,Word1,Word1_Category,Word2,Word2_Category,Count,Word1_occurrence,Word2_occurrence,Count_weighted
0,"(aandeel, energierekening)",aandeel,Overige,energierekening,Duurzaamheid,3,67,19,0.002357
1,"(aandeel, energie)",aandeel,Overige,energie,Duurzaamheid,51,67,373,0.002041
2,"(aandeel, klimaatakkoord)",aandeel,Overige,klimaatakkoord,Duurzaamheid,2,67,157,0.000190
3,"(aandeel, energieakkoord)",aandeel,Overige,energieakkoord,Duurzaamheid,1,67,23,0.000649
4,"(aandeel, energieverbruik)",aandeel,Overige,energieverbruik,Duurzaamheid,2,67,50,0.000597


In [18]:
relations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 772 entries, 0 to 771
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Word_Pair         772 non-null    object 
 1   Word1             772 non-null    object 
 2   Word1_Category    772 non-null    object 
 3   Word2             772 non-null    object 
 4   Word2_Category    772 non-null    object 
 5   Count             772 non-null    int64  
 6   Word1_occurrence  772 non-null    int64  
 7   Word2_occurrence  772 non-null    int64  
 8   Count_weighted    772 non-null    float64
dtypes: float64(1), int64(3), object(5)
memory usage: 54.4+ KB


In [19]:
df_data.head()

,Phrase,Page,Priority,word_pairs
0,"[integraal, nationaal, energie, klimaatplan, o...",6,1.0,"[(integraal, nationaal), (integraal, energie),..."
1,"[inek, bevat, hoofdlijnen, klimaat, energiebel...",6,1.0,"[(inek, bevat), (inek, hoofdlijnen), (inek, kl..."
2,"[inhoud, belangrijk, deel, bepaald, klimaatakk...",6,1.0,"[(inhoud, belangrijk), (inhoud, deel), (inhoud..."
3,"[akkoord, bevat, pakket, maatregelen, actieve,...",6,1.0,"[(akkoord, bevat), (akkoord, pakket), (akkoord..."
4,"[aanvulling, nieuwe, nationale, maatregelen, k...",6,1.0,"[(aanvulling, nieuwe), (aanvulling, nationale)..."


In [20]:
relations_df.shape

(772, 9)

In [21]:
relations_df.columns

Index(['Word_Pair', 'Word1', 'Word1_Category', 'Word2', 'Word2_Category',
       'Count', 'Word1_occurrence', 'Word2_occurrence', 'Count_weighted'],
      dtype='object')

In [22]:
# Change column names
# relations_df.columns = ['Word Pair', 'Source', 'Target', 'Count', 'Source_Occurrence', 'Target_Occurrence', 'Edge_Weight']
relations_df.columns = ['Word Pair', 'Source','Source_Category', 'Target','Target_Category', 'Count', 'Source_Occurrence', 'Target_Occurrence', 'Edge_Weight']

In [23]:
relations_df.head(50)

,Word Pair,Source,Source_Category,Target,Target_Category,Count,Source_Occurrence,Target_Occurrence,Edge_Weight
0,"(aandeel, energierekening)",aandeel,Overige,energierekening,Duurzaamheid,3,67,19,0.002357
1,"(aandeel, energie)",aandeel,Overige,energie,Duurzaamheid,51,67,373,0.002041
2,"(aandeel, klimaatakkoord)",aandeel,Overige,klimaatakkoord,Duurzaamheid,2,67,157,0.000190
3,"(aandeel, energieakkoord)",aandeel,Overige,energieakkoord,Duurzaamheid,1,67,23,0.000649
4,"(aandeel, energieverbruik)",aandeel,Overige,energieverbruik,Duurzaamheid,2,67,50,0.000597
5,"(aandeel, biobrandstoffen)",aandeel,Overige,biobrandstoffen,Duurzaamheid,2,67,26,0.001148
6,"(aantal, klimaatakkoord)",aantal,Overige,klimaatakkoord,Duurzaamheid,2,57,157,0.000223
7,"(aantal, energieverbruik)",aantal,Overige,energieverbruik,Duurzaamheid,1,57,50,0.000351
8,"(aardgas, energierekening)",aardgas,Overige,energierekening,Duurzaamheid,1,72,19,0.000731
9,"(aardgas, energie)",aardgas,Overige,energie,Duurzaamheid,3,72,373,0.000112


In [24]:
# Sort DataFrame by 'Weight' column in descending order
sorted_df = (relations_df.copy()).sort_values(by='Edge_Weight', ascending=False)

sorted_df.head(50)

,Word Pair,Source,Source_Category,Target,Target_Category,Count,Source_Occurrence,Target_Occurrence,Edge_Weight
730,"(variabiliteit, elektriciteitsopwekkingde)",variabiliteit,Duurzaamheid,elektriciteitsopwekkingde,Overige,1,1,1,1.000000
658,"(pblnotitie, klimaattransitie)",pblnotitie,Overige,klimaattransitie,Duurzaamheid,1,1,2,0.500000
710,"(statistieken, variabiliteit)",statistieken,Overige,variabiliteit,Duurzaamheid,1,2,1,0.500000
360,"(energiequote, betaalrisico)",energiequote,Duurzaamheid,betaalrisico,Overige,14,11,6,0.212121
578,"(klimaattransitie, update)",klimaattransitie,Duurzaamheid,update,Overige,1,2,3,0.166667
649,"(onzeker, variabiliteit)",onzeker,Overige,variabiliteit,Duurzaamheid,1,9,1,0.111111
579,"(klimaattransitie, april)",klimaattransitie,Duurzaamheid,april,Overige,1,2,6,0.083333
69,"(conventioneel, duurzaam)",conventioneel,Overige,duurzaam,Duurzaamheid,1,1,16,0.062500
14,"(afbakening, duurzaam)",afbakening,Overige,duurzaam,Duurzaamheid,1,1,16,0.062500
87,"(duurzaam, achtergrondrapportages)",duurzaam,Duurzaamheid,achtergrondrapportages,Overige,1,16,1,0.062500


In [25]:
# to csv
relations_df.to_csv('inek.csv', index=False)

In [26]:
# run GephiVisualization_nieuw notebook

%run Visualization_Final_Winnovation.ipynb


Community Leaders:
{3: 'beleid', 2: 'energie', 7: 'klimaatakkoord', 8: 'nr', 5: 'nederland', 1: 'elektriciteit', 0: 'co', 6: 'gas', 4: 'reductie'}
Theme/cluster: co: Red
Theme/cluster: elektriciteit: Yellow
Theme/cluster: energie: Cyan
Theme/cluster: beleid: Green
Theme/cluster: reductie: Pink
Theme/cluster: nederland: Blue
Theme/cluster: gas: Purple
Theme/cluster: klimaatakkoord: Light Pink
Theme/cluster: nr: Red
Cluster: co (Color: Red):
  Node: co, Node_Weight: 23.66
  Node: emissies, Node_Weight: 11.08
  Node: activiteiten, Node_Weight: 3.76
  Node: energy, Node_Weight: 2.47
  Node: kansen, Node_Weight: 2.26
  Node: overige, Node_Weight: 2.26
  Node: innovation, Node_Weight: 2.04
  Node: informatie, Node_Weight: 1.40
  Node: energieinnovatie, Node_Weight: 1.18
  Node: restwarmte, Node_Weight: 0.65
  Node: geleverde, Node_Weight: 0.54
  Node: clean, Node_Weight: 0.32
  Node: dei, Node_Weight: 0.11

Cluster: elektriciteit (Color: Yellow):
  Node: elektriciteit, Node_Weight: 14.95
  N